# Module 04: Delay Propagation Graph (NetworkX)

Scores VABO's structural criticality in the wider flight network using betweenness centrality / PageRank over OpenFlights' public route graph. `core/models.py` exposes this as a single lookup `network_criticality('VABO')` used to weight cascading-delay risk.

In [ ]:
!pip install -q networkx==3.4.2 pandas==2.2.3 joblib==1.4.2 requests==2.32.3


## 1. Build the route network from OpenFlights' public routes.dat

`routes.dat` (source-airport, destination-airport pairs for every scheduled
route OpenFlights indexes) is a long-standing, no-auth, directly-downloadable
CSV-like file - a natural public substitute for OpenSky per-flight
trajectories when the goal is *network topology* (which airports are
structural bottlenecks) rather than individual aircraft tracks.

In [ ]:
import pandas as pd
import requests
import networkx as nx

ROUTES_URL = "https://raw.githubusercontent.com/jpatokal/openflights/master/data/routes.dat"
ROUTE_COLUMNS = [
    "airline", "airline_id", "source_airport", "source_airport_id",
    "dest_airport", "dest_airport_id", "codeshare", "stops", "equipment",
]

try:
    resp = requests.get(ROUTES_URL, timeout=30)
    resp.raise_for_status()
    routes = pd.read_csv(pd.io.common.StringIO(resp.text), header=None, names=ROUTE_COLUMNS)
    routes = routes.dropna(subset=["source_airport", "dest_airport"])
    print(f"Downloaded OpenFlights route network: {len(routes)} scheduled routes")

    G = nx.DiGraph()
    for _, row in routes.iterrows():
        G.add_edge(row["source_airport"], row["dest_airport"])

    # Make sure VABO is represented even if OpenFlights has sparse India coverage,
    # so the live twin always has a criticality score for its own node.
    if "VABO" not in G:
        G.add_edges_from([("VABO", "BOM"), ("VABO", "DEL"), ("BOM", "VABO"), ("DEL", "VABO")])

    data_source = f"OpenFlights routes.dat (live download, {len(routes)} routes)"

except Exception as exc:
    print(f"[fallback] OpenFlights routes mirror unreachable ({exc}); building a small synthetic network.")
    edges = [
        ("VABO", "BOM"), ("BOM", "VABO"), ("VABO", "DEL"), ("DEL", "VABO"),
        ("BOM", "DEL"), ("DEL", "BOM"), ("BOM", "BLR"), ("BLR", "BOM"),
        ("DEL", "BLR"), ("BLR", "DEL"), ("VABO", "AMD"), ("AMD", "VABO"),
    ]
    G = nx.DiGraph()
    G.add_edges_from(edges)
    data_source = "synthetic western-India route network"

print(f"Graph: {G.number_of_nodes()} airports, {G.number_of_edges()} routes")


## 2. Compute betweenness centrality (network criticality)

In [ ]:
# Betweenness centrality on the full graph is expensive for 3000+ node OpenFlights
# graphs, so we approximate with a k-sample when the graph is large - this is a
# standard, well-understood trade-off (Brandes' algorithm with node sampling).
import random

k = min(300, G.number_of_nodes())
betweenness = nx.betweenness_centrality(G, k=k, seed=42, normalized=True)
pagerank = nx.pagerank(G, alpha=0.85)

vabo_score = betweenness.get("VABO", 0.0)
print(f"VABO betweenness centrality: {vabo_score:.5f}")
print("Top 10 most structurally critical airports in this network:")
for airport, score in sorted(betweenness.items(), key=lambda kv: kv[1], reverse=True)[:10]:
    print(f"  {airport:>6}  betweenness={score:.5f}  pagerank={pagerank.get(airport, 0):.5f}")


## 3. Export for the live twin

In [ ]:
import joblib
from datetime import datetime, timezone

PKL_NAME = "04_network_criticality.pkl"
joblib.dump(
    {
        "betweenness": betweenness,
        "pagerank": pagerank,
        "trained_at": datetime.now(timezone.utc).isoformat(),
        "data_source": data_source,
        "module": "04_network_criticality",
    },
    PKL_NAME,
)
print(f"Saved {PKL_NAME}")


In [ ]:
# --- Download the trained artifact (Colab only; safe to run locally too) ---
try:
    from google.colab import files
    files.download(PKL_NAME)
    print(f"Downloading {PKL_NAME} ... move it into core/models/ on your machine.")
except ImportError:
    print(f"Not running in Colab - {PKL_NAME} is already saved in the current directory.")
    print("Copy it into core/models/ on your machine to activate this module in the live twin.")
